# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [8]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [9]:
@function_tool
def get_food_nutrients(food_item: str) -> str:
    """
    Get approximate macronutrient information for common foods.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Protein, carbohydrates, fat, and fiber per standard serving
    """
    nutrient_data = {
        "apple": "per medium apple (182g): 0.5g protein, 25g carbohydrates, 0.3g fat, 4.4g fiber",
        "banana": "per medium banana (118g): 1.3g protein, 27g carbohydrates, 0.4g fat, 3.1g fiber",
        "broccoli": "per 1 cup chopped (91g): 2.5g protein, 6g carbohydrates, 0.3g fat, 2.4g fiber",
        "almonds": "per 1oz (28g): 6g protein, 6.1g carbohydrates, 14g fat, 3.5g fiber",
    }

    food_key = food_item.lower()
    if food_key in nutrient_data:
        return f"{food_item.title()}: {nutrient_data[food_key]} (approximate)"
    return f"I don't have nutrient data for {food_item} in my database. Try common foods like apple, banana, broccoli, or almonds."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [5]:
get_food_calories('banana')

TypeError: 'FunctionTool' object is not callable

In [10]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories,get_food_nutrients])

In [14]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories and nutrients are in total in an litchi?"
    )
    print(result.final_output)

Lychee (per 100 g, approximate)
- Calories: ~66 kcal
- Carbs: ~16.5 g (sugars ~15 g)
- Protein: ~0.8 g
- Fat: ~0.4 g
- Fiber: ~1.1 g
- Vitamin C: ~71 mg
- Potassium: ~171 mg

One lychee (about 8–10 g) ≈ 5–7 kcal, with negligible fat and protein. Values can vary by size and variety.


Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)